In [1]:
import pandas as pd
data = pd.read_csv("../data/raw/Bank-Customer-Churn-Prediction.csv")

In [9]:
data.dtypes

credit_score          int64
country              object
gender               object
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card           int64
active_member         int64
estimated_salary    float64
churn                 int64
dtype: object

In [3]:
#checking missing values
data.isnull().sum()

credit_score        0
country             0
gender              0
age                 0
tenure              0
balance             0
products_number     0
credit_card         0
active_member       0
estimated_salary    0
churn               0
dtype: int64

In [4]:
#checking for duplicates
data.duplicated().sum()

np.int64(0)

In [7]:
#checking categorial columns
categorical_cols = data.select_dtypes(include=['object']).columns.tolist()
print("Categorical columns:", categorical_cols)

# See unique values
for col in categorical_cols:
    print(f"\n{col}: {data[col].unique()}")

Categorical columns: ['country', 'gender']

country: ['France' 'Spain' 'Germany']

gender: ['Female' 'Male']


In [14]:
from sklearn.preprocessing import OneHotEncoder

# Create OneHotEncoder
encoder = OneHotEncoder(drop='first', sparse_output=False)

# Fit and transform the categorical columns
encoded_features = encoder.fit_transform(data[categorical_cols])

# Get feature names
encoded_feature_names = encoder.get_feature_names_out(categorical_cols)

# Create dataframe with encoded features
data_encoded_features = pd.DataFrame(encoded_features, columns=encoded_feature_names)

# Drop original categorical columns and add encoded ones
data_encoded = data.drop(columns=categorical_cols)
data_encoded = pd.concat([data_encoded, data_encoded_features], axis=1)

In [12]:
# Verify all columns are numeric
data_encoded.dtypes

credit_score          int64
age                   int64
tenure                int64
balance             float64
products_number       int64
credit_card           int64
active_member         int64
estimated_salary    float64
churn                 int64
country_Germany     float64
country_Spain       float64
gender_Male         float64
dtype: object

Adding new features to avoid the case where models perform poorly (<0.70 ROC-AUC):

New feautures were based on boxplot results (01_eda.ipynb): [age, balance, tenure]
Correlation results showed a weak negative correlation between [churn, active_member]

High Priority (Strongest correlations):
Age (0.285) - Age groups, age interactions
Active_member (-0.156) - Interactions with products, balance, age
Balance (0.119) - Has_balance flag, balance categories, interactions

Medium Priority (Weaker but still useful):
Tenure (-0.014) - Tenure groups (might reveal non-linear patterns)
Products_number (-0.048) - Interactions with active_member

New features helps to:
Capture non-linear relationships (age groups, tenure groups)
Highlight important thresholds (has_balance)
Combine weak signals into stronger ones (active_products)
Make patterns easier for models to find

In [ ]:
#Age groups
data_encoded['age_group'] = pd.cut(data_encoded['age'], 
                                 bins=[0, 30, 40, 50, 60, 100], 
                                 labels=['young', 'middle', 'senior', 'old', 'very_old'])
data_encoded = pd.get_dummies(data_encoded, columns=['age_group'], drop_first=True)

In [ ]:
#Balance flag (has balance or not)
data_encoded['has_balance'] = (data_encoded['balance'] > 0).astype(int)

In [ ]:
#Tenure groups
data_encoded['tenure_group'] = pd.cut(data_encoded['tenure'], 
                                    bins=[-1, 2, 5, 10],
                                    labels=['new', 'medium', 'loyal'])
data_encoded = pd.get_dummies(data_encoded, columns=['tenure_group'], drop_first=True)

In [ ]:
#Active member interaction
data_encoded['active_products'] = data_encoded['active_member'] * data_encoded['products_number']

In [ ]:
# Verification
data_encoded.shape
data_encoded.columns.tolist()


['credit_score',
 'age',
 'tenure',
 'balance',
 'products_number',
 'credit_card',
 'active_member',
 'estimated_salary',
 'churn',
 'country_Germany',
 'country_Spain',
 'gender_Male',
 'age_group_middle',
 'age_group_senior',
 'age_group_old',
 'age_group_very_old',
 'has_balance',
 'tenure_group_medium',
 'tenure_group_loyal',
 'active_products']

In [ ]:
# Save
data_encoded.to_csv("C:/Users/hp/churn-prediction/data/processed/churn_dataset_clean.csv", index=False)
print("Saved!")

Saved!
